# FMCG Training & Multi-Horizon Forecasting Pipeline

**Production Version — Rich Features + Memory Optimized**

This notebook trains three LightGBM models (horizons h=1, h=7, h=14 days) on FMCG inventory time-series data.
It covers the full ML lifecycle:

| Step | Description |
|------|-------------|
| 1 | Load & memory-optimise raw parquet data |
| 2 | Stockout correction (censored demand recovery) |
| 3 | Rich feature engineering (calendar, lags, weather, festivals, shocks, competitors) |
| 4 | Multi-horizon model training with early stopping |
| 5 | Evaluation (WMAPE, MAE, RMSE, R², Bias) |
| 6 | SHAP explainability |
| 7 | Drift baseline capture |

> **Primary metric:** WMAPE (Weighted Mean Absolute Percentage Error).  
> Unlike MAPE, WMAPE weights errors by volume — so near-zero SKUs don't distort the score.


## 1. Setup & Imports

Install any missing dependencies, then import all required libraries.
- `lightgbm` — gradient boosting framework (fast, handles missing values natively)
- `shap` — model explainability (optional; gracefully skipped if unavailable)
- `psutil` — memory monitoring
- `matplotlib` — plotting (non-interactive `Agg` backend for server/Kaggle compatibility)


In [1]:
# Install dependencies if running on Kaggle or a fresh environment
# !pip install lightgbm shap psutil -q

import logging
import gc
import json
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Optional: SHAP
try:
    import shap
    SHAP_AVAILABLE = True
    logger.info('SHAP available')
except ImportError:
    SHAP_AVAILABLE = False
    logger.warning('SHAP not available — explainability plots will be skipped')

import matplotlib
matplotlib.use('Agg')  # Non-interactive backend — safe for Kaggle/servers
import matplotlib.pyplot as plt

print('All imports successful.')


2026-05-05 03:12:28,368 - INFO - SHAP available


All imports successful.


## 2. Configuration

All paths and constants are centralised here so nothing is hardcoded deeper in the pipeline.

- **`DATA_ROOT`** auto-detects Kaggle (`/kaggle/input/fmcgparquet`) vs local (`./data`)
- **`LEAK_COLS`** — columns derived from the target that would cause data leakage if used as features
- **`HORIZONS`** — forecast horizons in days: 1-day, 7-day, 14-day
- **`SHAP_SAMPLE_TEST`** — SHAP is expensive; we sample 500 rows to keep it fast


In [2]:
# ── Path Detection ──────────────────────────────────────────────────────
if Path('/kaggle/input').exists():
    DATA_ROOT   = Path('/kaggle/input/datasets/sunchoehprince/fmcgparquet/')
    OUTPUT_ROOT = Path('/kaggle/working')
else:
    DATA_ROOT   = Path('./data')
    OUTPUT_ROOT = Path('./output')
OUTPUT_ROOT.mkdir(exist_ok=True)

# ── File Names ───────────────────────────────────────────────────────────
DAILY_TS_FILE = 'daily_timeseries.parquet'
SKU_FILE      = 'sku_master.parquet'
LOCATION_FILE = 'location_master.parquet'
FESTIVAL_FILE = 'festival_calendar.parquet'
WEATHER_FILE  = 'weather_data.parquet'
MACRO_FILE    = 'macro_indicators.parquet'
SHOCKS_FILE   = 'external_shocks_daily.parquet'
COMP_FILE     = 'competitor_activity.parquet'

TARGET_COL = 'true_demand'

# Columns that would leak future/target information — always excluded from features
LEAK_COLS = [
    'expected_demand', 'unfulfilled_demand',
    'reorder_point', 'safety_stock',
    'actual_demand', 'unconstrained_demand'
]

# ── Training Flags ───────────────────────────────────────────────────────
SHAP_SAMPLE_TEST = 500   # rows sampled for SHAP (full dataset is too slow)
ENABLE_SHAP      = SHAP_AVAILABLE
HORIZONS         = [1, 7, 14]  # forecast horizons in days

print(f'DATA_ROOT   : {DATA_ROOT}')
print(f'OUTPUT_ROOT : {OUTPUT_ROOT}')
print(f'Horizons    : {HORIZONS}')


DATA_ROOT   : /kaggle/input/datasets/sunchoehprince/fmcgparquet
OUTPUT_ROOT : /kaggle/working
Horizons    : [1, 7, 14]


## 3. Memory Utilities

Kaggle notebooks have a ~16 GB RAM cap. With millions of rows across multiple horizons,
memory management is critical.

### `reduce_mem_usage`
Downcasts numeric columns to the smallest type that fits the data range:
- `int64` → `int8/int16/int32` (saves 4–8×)
- `float64` → `float32` (saves 2×, sufficient precision for demand forecasting)

### `get_memory_usage`
Returns current process RSS in MB — used as checkpoints throughout the pipeline.


In [3]:
def get_memory_usage():
    import os, psutil
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2


def reduce_mem_usage(df, verbose=False):
    """Downcast numeric columns to smallest safe dtype."""
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if 'datetime' in str(col_type) or 'category' in str(col_type):
            continue
        if str(col_type)[:3] == 'int':
            c_min, c_max = df[col].min(), df[col].max()
            if   c_min > np.iinfo(np.int8).min  and c_max < np.iinfo(np.int8).max:  df[col] = df[col].astype(np.int8)
            elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max: df[col] = df[col].astype(np.int16)
            elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max: df[col] = df[col].astype(np.int32)
        elif str(col_type)[:5] == 'float':
            c_min, c_max = df[col].min(), df[col].max()
            if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                df[col] = df[col].astype(np.float32)
    if verbose:
        end_mem = df.memory_usage().sum() / 1024**2
        logger.info(f'Memory: {start_mem:.2f} MB → {end_mem:.2f} MB ({100*(start_mem-end_mem)/start_mem:.1f}% reduction)')
    return df

print('Memory utilities defined.')


Memory utilities defined.


## 4. Evaluation Metrics

### Why WMAPE?
Standard MAPE breaks on near-zero demand (division by ~0 → infinite error).
WMAPE avoids this by dividing by the **sum** of actuals rather than each individual value:

$$\text{WMAPE} = \frac{\sum |y_i - \hat{y}_i|}{\sum |y_i|} \times 100$$

This naturally down-weights low-volume SKUs and up-weights high-volume ones — exactly what
a business cares about (a 50% error on a SKU selling 2 units/day matters far less than
a 10% error on one selling 2,000 units/day).

### Full Metrics Suite
| Metric | Purpose |
|--------|---------|
| WMAPE | Primary — volume-weighted accuracy |
| MAE | Average absolute error in units |
| RMSE | Penalises large errors more heavily |
| MAPE | Percentage error (non-zero rows only) |
| R² | Variance explained |
| Bias | Mean signed error (+ve = under-forecast) |
| Over/Under % | Directional bias breakdown |


In [4]:
def calculate_metrics(y_true, y_pred, label=''):
    """Compute full metrics suite. WMAPE is the primary KPI."""
    y_pred = np.clip(y_pred, 0, None)  # demand can't be negative
    y_true, y_pred = np.array(y_true), np.array(y_pred)

    wmape = np.sum(np.abs(y_true - y_pred)) / max(np.sum(np.abs(y_true)), 1) * 100
    mae   = np.mean(np.abs(y_true - y_pred))
    rmse  = np.sqrt(np.mean((y_true - y_pred) ** 2))
    bias  = np.mean(y_true - y_pred)  # positive = under-forecast

    nonzero = y_true > 0
    mape = np.mean(np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero])) * 100 if nonzero.sum() > 0 else 0.0

    denom = np.sum((y_true - y_true.mean()) ** 2)
    r2 = 1 - np.sum((y_true - y_pred) ** 2) / denom if denom > 0 else 0.0

    over_pct  = (y_pred > y_true).mean() * 100
    under_pct = (y_pred < y_true).mean() * 100

    if label:
        logger.info(f'--- Metrics: {label} ---')
    logger.info(f'  WMAPE:  {wmape:.2f}%  (primary metric)')
    logger.info(f'  MAE:    {mae:.2f}')
    logger.info(f'  RMSE:   {rmse:.2f}')
    logger.info(f'  MAPE:   {mape:.2f}%  (non-zero rows only)')
    logger.info(f'  R\u00b2:     {r2:.4f}')
    logger.info(f'  Bias:   {bias:+.2f}  (+ = under-forecast)')
    logger.info(f'  Over:   {over_pct:.1f}%  Under: {under_pct:.1f}%')

    return {'wmape': wmape, 'mae': mae, 'rmse': rmse, 'mape': mape, 'r2': r2, 'bias': bias}

print('Metrics functions defined.')


Metrics functions defined.


## 5. Stockout Correction

### The Problem: Censored Demand
When a product is out of stock, `actual_demand` records only what was *sold*, not what customers *wanted*.
Training on censored demand teaches the model that stockout days have low demand — the opposite of reality.

### The Solution: Two-tier correction

**Fully censored rows** (opening stock = 0, no incoming stock, stockout flag = 1):
- These rows are **dropped entirely** — we have no signal at all about true demand.

**Partially censored rows** (stockout occurred but some sales recorded):
- Replace `actual_demand` with `max(actual_demand, rolling_velocity)`
- `rolling_velocity` = 7-day rolling mean of prior demand (capped at p99 to avoid outlier inflation)
- This recovers the *minimum plausible* true demand

> **Why p99 cap instead of p95?**  
> p95 was clipping genuine high-demand peaks (festival days, promotions). p99 is more conservative.


In [5]:
def correct_stockouts(df):
    logger.info('Applying stockout correction...')
    if 'stockout_flag' not in df.columns:
        df['true_demand'] = df['actual_demand'].astype(np.float32)
        return df

    df = df.sort_values(['sku_id', 'location_id', 'date']).reset_index(drop=True)

    # Rolling velocity: 7-day mean of prior demand (shift(1) avoids same-day leakage)
    df['_velocity'] = df.groupby(['sku_id', 'location_id'], observed=True)['actual_demand'].transform(
        lambda x: x.shift(1).rolling(7, min_periods=3).mean()
    ).astype(np.float32)

    # Cap velocity at p99 per SKU-location to avoid outlier inflation
    df['_velocity_p99'] = df.groupby(['sku_id', 'location_id'], observed=True)['_velocity'].transform(
        lambda x: x.quantile(0.99)
    )
    df['_velocity'] = np.minimum(df['_velocity'].values, df['_velocity_p99'].values)

    # Fully censored: zero stock in, zero stock on hand, stockout flag set
    fully_censored = (
        (df.get('opening_stock',  pd.Series(1, index=df.index)) == 0) &
        (df.get('incoming_stock', pd.Series(1, index=df.index)) == 0) &
        (df.get('stockout_flag',  pd.Series(0, index=df.index)) == 1)
    )
    # Partially censored: stockout occurred but some sales were recorded
    partially_censored = (
        (df['stockout_flag'] == 1) &
        (df.get('closing_stock', 1) == 0) &
        (df['actual_demand'] > 0)
    )

    df['true_demand'] = df['actual_demand'].astype(np.float32)
    df.loc[partially_censored, 'true_demand'] = np.maximum(
        df.loc[partially_censored, 'actual_demand'].values,
        df.loc[partially_censored, '_velocity'].values
    )

    df = df[~fully_censored].reset_index(drop=True)
    df = df.drop(columns=['_velocity', '_velocity_p99'], errors='ignore')

    logger.info(f'Excluded {fully_censored.sum():,} fully censored rows')
    gc.collect()
    return df

print('Stockout correction defined.')


Stockout correction defined.


## 6. Feature Engineering

Features are built from **11 sources**. Each is only merged if not already present
(safe to re-run without duplication).

| # | Source | Features Added |
|---|--------|----------------|
| 1 | SKU Master | category, lifecycle_stage, days_since_birth |
| 2 | Location Master | population_base, store_type, region |
| 3 | Festival Calendar | festival_multiplier, festival_duration |
| 4 | Weather | temperature, rainfall, humidity, etc. |
| 5 | ~~Macro Indicators~~ | *Skipped — monthly grain incompatible with daily model* |
| 6 | External Shocks | shock_active, shock_demand_impact, shock_supply_impact |
| 7 | Competitor Activity | competitor_promo_flag, promo_intensity, price_pressure |
| 8 | Calendar | day_of_week, month, is_weekend, cyclical sin/cos encodings |
| 9 | Price & Promo | discount_depth, price_vs_base |
| 10 | Channel | channel_enc (ordinal encoding) |
| 11 | Lag & Rolling | demand_lag_1/7/14/30, rolling_mean/std 7/14/30d |

### Why cyclical encoding for calendar features?
Day 0 (Monday) and Day 6 (Sunday) are adjacent in the week, but a raw integer encoding
treats them as maximally distant (0 vs 6). Sin/cos encoding preserves the circular structure:
$$\text{dow\_sin} = \sin\left(\frac{2\pi \cdot \text{day\_of\_week}}{7}\right)$$

### Why skip Macro Indicators?
GDP/CPI are monthly. Forward-filling to daily creates 30 identical rows per month —
LightGBM sees no within-month variation, so feature importance → ~0. Re-enable if
the model is ever aggregated to weekly/monthly grain.

### Lag feature leakage prevention
All lags use `shift(1)` before rolling — this ensures the model never sees same-day
or future demand during training.


In [6]:
def add_calendar_features(df):
    df['day_of_week']     = df['date'].dt.weekday.astype(np.int8)
    df['day_of_month']    = df['date'].dt.day.astype(np.int8)
    df['month']           = df['date'].dt.month.astype(np.int8)
    df['is_weekend']      = (df['day_of_week'] >= 5).astype(np.int8)
    # Cyclical encoding preserves the circular nature of weekdays
    df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7).astype(np.float32)
    df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7).astype(np.float32)
    return df


def add_lag_features(df, target_col=TARGET_COL):
    logger.info('Creating lag features...')
    df = df.sort_values(['sku_id', 'location_id', 'date']).reset_index(drop=True)
    g = df.groupby(['sku_id', 'location_id'], observed=True)

    # Point-in-time lags — shift(1) ensures no same-day leakage
    for lag in [1, 7, 14, 30]:
        df[f'demand_lag_{lag}'] = g[target_col].shift(lag).astype(np.float32)

    # Rolling statistics — shift(1) before rolling to avoid leakage
    shifted = g[target_col].shift(1)
    for window in [7, 14, 30]:
        df[f'demand_rolling_{window}_mean'] = shifted.rolling(window, min_periods=1).mean().astype(np.float32)
        df[f'demand_rolling_{window}_std']  = shifted.rolling(window, min_periods=1).std().fillna(0).astype(np.float32)

    # Promo intensity: fraction of promo days in last 7 days
    if 'promo_flag' in df.columns:
        df['promo_intensity_7d'] = g['promo_flag'].shift(1).rolling(7, min_periods=1).mean().astype(np.float32)

    return df

print('Calendar and lag feature functions defined.')


Calendar and lag feature functions defined.


In [7]:
def engineer_features(df):
    """Full feature engineering pipeline — merges all available data sources."""
    logger.info('Starting feature engineering...')

    # 1. SKU Master
    if 'category' not in df.columns:
        sku_path = DATA_ROOT / SKU_FILE
        if sku_path.exists():
            try:
                sku_df = pd.read_parquet(sku_path)
                df = df.merge(sku_df, on='sku_id', how='left', suffixes=('', '_sku'))
                if 'birth_date' in df.columns:
                    df['days_since_birth'] = (df['date'] - pd.to_datetime(df['birth_date'])).dt.days.clip(lower=0).astype(np.int16)
                    df = df.drop(columns=['birth_date', 'retirement_date', 'sku_name', 'pack_size', 'supplier_id'], errors='ignore')
                logger.info(f'Merged SKU master ({len(sku_df)} rows)')
            except Exception as e:
                logger.warning(f'SKU master merge failed: {e}')
    else:
        logger.info('SKU fields already present — skipping')

    # 2. Location Master
    if 'population_base' not in df.columns:
        loc_path = DATA_ROOT / LOCATION_FILE
        if loc_path.exists():
            try:
                loc_df = pd.read_parquet(loc_path)
                df = df.merge(loc_df, on='location_id', how='left', suffixes=('', '_loc'))
                df = df.drop(columns=['city'], errors='ignore')
                logger.info(f'Merged Location master ({len(loc_df)} rows)')
            except Exception as e:
                logger.warning(f'Location master merge failed: {e}')
    else:
        logger.info('Location fields already present — skipping')

    # 3. Festival Calendar — demand_multiplier + duration (richer than binary flag)
    fest_path = DATA_ROOT / FESTIVAL_FILE
    if fest_path.exists():
        try:
            fest_df = pd.read_parquet(fest_path)[['date', 'demand_multiplier', 'festival_duration_days']]
            fest_df = fest_df.groupby('date').agg(
                festival_multiplier=('demand_multiplier', 'max'),
                festival_duration=('festival_duration_days', 'max')
            ).reset_index()
            df = df.merge(fest_df, on='date', how='left')
            df['festival_multiplier'] = df['festival_multiplier'].fillna(1.0).astype(np.float32)
            df['festival_duration']   = df['festival_duration'].fillna(0).astype(np.int8)
            logger.info('Added festival features')
        except Exception as e:
            logger.warning(f'Festival merge failed: {e}')

    # 4. Weather
    weather_path = DATA_ROOT / WEATHER_FILE
    if weather_path.exists():
        try:
            weather_df = pd.read_parquet(weather_path)
            df = df.merge(weather_df, on='date', how='left')
            for c in weather_df.columns.drop('date'):
                if c in df.columns:
                    df[c] = df[c].astype(np.float32)
            logger.info(f'Added weather features ({len(weather_df.columns)-1} fields)')
        except Exception as e:
            logger.warning(f'Weather merge failed: {e}')

    # 5. Macro Indicators — intentionally skipped (monthly grain incompatible with daily model)
    logger.info('Skipping macro indicators (monthly grain incompatible with daily model)')

    # 6. External Shocks
    shocks_path = DATA_ROOT / SHOCKS_FILE
    if shocks_path.exists():
        try:
            shocks_df = pd.read_parquet(shocks_path)[['date', 'shock_active', 'shock_demand_impact', 'shock_supply_impact']]
            df = df.merge(shocks_df, on='date', how='left')
            df['shock_active']        = df['shock_active'].fillna(0).astype(np.int8)
            df['shock_demand_impact'] = df['shock_demand_impact'].fillna(1.0).astype(np.float32)
            df['shock_supply_impact'] = df['shock_supply_impact'].fillna(1.0).astype(np.float32)
            logger.info('Added external shock features')
        except Exception as e:
            logger.warning(f'Shocks merge failed: {e}')

    # 7. Competitor Activity
    comp_path = DATA_ROOT / COMP_FILE
    if comp_path.exists():
        try:
            comp_df = pd.read_parquet(comp_path)
            date_col = 'date' if 'date' in comp_df.columns else 'month'
            comp_df = comp_df.rename(columns={date_col: 'date'})
            comp_df['date'] = pd.to_datetime(comp_df['date'])
            join_keys = ['date', 'category'] if ('category' in comp_df.columns and 'category' in df.columns) else ['date']
            agg_cols = {c: 'mean' for c in ['competitor_promo_intensity', 'competitor_price_pressure'] if c in comp_df.columns}
            flag_cols = {c: 'max' for c in ['competitor_promo_flag'] if c in comp_df.columns}
            agg_cols.update(flag_cols)
            if agg_cols:
                comp_df = comp_df.groupby(join_keys).agg(agg_cols).reset_index()
                df = df.merge(comp_df, on=join_keys, how='left')
                for c, default in [('competitor_promo_flag', 0), ('competitor_promo_intensity', 0.0), ('competitor_price_pressure', 1.0)]:
                    if c in df.columns:
                        df[c] = df[c].fillna(default).astype(np.float32)
                logger.info(f'Added competitor features (joined on {join_keys})')
        except Exception as e:
            logger.warning(f'Competitor merge failed: {e}')

    # 8. Calendar Features
    if 'day_of_week' not in df.columns:
        df = add_calendar_features(df)
        df['week_of_year'] = df['date'].dt.isocalendar().week.astype(np.int8)
        df['quarter']      = df['date'].dt.quarter.astype(np.int8)
    df['month_sin'] = np.sin(2 * np.pi * df['date'].dt.month / 12).astype(np.float32)
    df['month_cos'] = np.cos(2 * np.pi * df['date'].dt.month / 12).astype(np.float32)

    # 9. Price & Promo
    if 'base_price' in df.columns and 'price' in df.columns:
        df['discount_depth'] = ((df['base_price'] - df['price']) / df['base_price'].replace(0, np.nan)).fillna(0).clip(0, 1).astype(np.float32)
        df['price_vs_base']  = (df['price'] / df['base_price'].replace(0, np.nan)).fillna(1.0).astype(np.float32)

    # 10. Channel encoding
    if 'channel' in df.columns:
        channel_map = {'ModernTrade': 0, 'Traditional': 1, 'Ecommerce': 2}
        df['channel_enc'] = df['channel'].map(channel_map).fillna(-1).astype(np.int8)

    # 11. Lag Features
    if 'demand_lag_1' not in df.columns:
        df = add_lag_features(df)
    else:
        logger.info('Lag features already present — skipping rebuild')

    # Drop rows missing core lags (first ~30 days per SKU-location)
    df = df.dropna(subset=['demand_lag_1', 'demand_lag_7']).reset_index(drop=True)

    logger.info(f'Feature engineering complete. Shape: {df.shape}')
    gc.collect()
    return df

print('Feature engineering pipeline defined.')


Feature engineering pipeline defined.


## 7. Drift Detection

Model drift occurs when the statistical properties of production data diverge from training data.
This is especially common in FMCG due to:
- Seasonal demand shifts
- New product launches / discontinuations
- Supply chain disruptions
- Competitor promotions

### How it works
1. **Capture baseline** — record mean/std of each feature from the training set
2. **Save to JSON** — persisted as `drift_baseline.json` in the output directory
3. **Production monitoring** — the backend (`drift_retraining.py`) loads this baseline
   and computes z-scores on incoming data. If drift exceeds `drift_threshold` (default 15%),
   it triggers automatic retraining via the Kaggle API.

> The baseline is captured from the **h=1 training split** — the most recent and representative
> slice of data. It is captured *before* that split is deleted to free memory.


In [8]:
class DriftDetector:
    """Captures training distribution statistics for production drift monitoring."""

    def __init__(self, drift_threshold=0.15):
        self.drift_threshold = drift_threshold
        self.baseline_stats  = {}
        self.critical_features = []

    def capture_baseline(self, X_train, y_train, critical_features=None):
        logger.info('Capturing baseline statistics...')
        self.critical_features = critical_features if critical_features else X_train.columns.tolist()[:20]

        for col in self.critical_features:
            if col in X_train.columns:
                values = X_train[col].dropna()
                self.baseline_stats[col] = {
                    'mean': float(values.mean()),
                    'std':  float(values.std()) if values.std() > 0 else 1.0
                }

        self.baseline_stats['__target__'] = {
            'mean': float(y_train.mean()),
            'std':  float(y_train.std()) if y_train.std() > 0 else 1.0
        }
        logger.info(f'\u2713 Baseline captured for {len(self.baseline_stats)} features')

    def save_baseline(self, filepath):
        baseline_data = {
            'baseline_stats':    self.baseline_stats,
            'critical_features': self.critical_features,
            'drift_threshold':   self.drift_threshold,
            'timestamp':         datetime.now().isoformat()
        }
        with open(filepath, 'w') as f:
            json.dump(baseline_data, f, indent=2)
        logger.info(f'\u2713 Baseline saved to {filepath}')

print('DriftDetector defined.')


DriftDetector defined.


## 8. Performance Visualisation

Five diagnostic plots are generated after training:

| Plot | File | Description |
|------|------|-------------|
| 1 | `perf_01_wmape_by_horizon.png` | WMAPE bar chart across H1/H7/H14 |
| 2 | `perf_02_metrics_by_horizon.png` | WMAPE, MAE, RMSE side-by-side |
| 3 | `perf_03_actual_vs_predicted_h1.png` | Scatter plot — H1 actual vs predicted |
| 4 | `perf_04_residuals_h1.png` | Error distribution histogram |
| 5 | `perf_05_bias_and_r2.png` | Bias direction + R² by horizon |

All plots use a dark theme consistent with the dashboard UI.
Saved to `OUTPUT_ROOT/performance_plots/`.


In [9]:
def plot_model_performance(all_results, horizons, output_dir):
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)

    STYLE = {
        'figure.facecolor': '#0f172a', 'axes.facecolor': '#1e293b',
        'axes.edgecolor': '#334155',   'axes.labelcolor': '#94a3b8',
        'xtick.color': '#94a3b8',      'ytick.color': '#94a3b8',
        'text.color': '#e2e8f0',       'grid.color': '#334155',
        'grid.alpha': 0.5,             'axes.titlecolor': '#f1f5f9'
    }
    plt.rcParams.update(STYLE)
    BLUE, GREEN, ORANGE, PURPLE = '#38bdf8', '#34d399', '#fb923c', '#818cf8'

    # Plot 1: WMAPE by horizon
    fig, ax = plt.subplots(figsize=(8, 5))
    wmapes = [all_results[h]['metrics']['wmape'] for h in horizons]
    ax.bar([f'H{h}' for h in horizons], wmapes, color=BLUE, alpha=0.85, width=0.4)
    ax.set_title('WMAPE by Forecast Horizon', fontsize=14, pad=12)
    ax.set_ylabel('WMAPE (%)')
    ax.grid(axis='y')
    for i, v in enumerate(wmapes):
        ax.text(i, v + 0.2, f'{v:.1f}%', ha='center', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(output_path / 'perf_01_wmape_by_horizon.png', dpi=150, bbox_inches='tight')
    plt.close()

    # Plot 2: All metrics by horizon
    metric_names  = ['wmape', 'mae', 'rmse']
    metric_colors = [BLUE, GREEN, ORANGE]
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    for ax, metric, color in zip(axes, metric_names, metric_colors):
        vals = [all_results[h]['metrics'][metric] for h in horizons]
        ax.bar([f'H{h}' for h in horizons], vals, color=color, alpha=0.85, width=0.4)
        ax.set_title(metric.upper(), fontsize=13)
        ax.grid(axis='y')
        for i, v in enumerate(vals):
            ax.text(i, v * 1.01, f'{v:.1f}', ha='center', fontsize=10)
    plt.suptitle('Model Performance by Forecast Horizon', fontsize=14)
    plt.tight_layout()
    plt.savefig(output_path / 'perf_02_metrics_by_horizon.png', dpi=150, bbox_inches='tight')
    plt.close()

    # Plot 3: Actual vs Predicted scatter (H1)
    if 'y_test' in all_results[1]:
        y_true = all_results[1]['y_test']
        y_pred = all_results[1]['y_pred']
        nonzero = y_true > 0
        y_true_nz, y_pred_nz = y_true[nonzero], y_pred[nonzero]
        n_sample = min(3000, len(y_true_nz))
        idx = np.random.choice(len(y_true_nz), n_sample, replace=False)
        fig, ax = plt.subplots(figsize=(7, 7))
        ax.scatter(y_true_nz[idx], y_pred_nz[idx], alpha=0.25, s=8, color=BLUE)
        lim = max(y_true_nz[idx].max(), y_pred_nz[idx].max())
        ax.plot([0, lim], [0, lim], color=ORANGE, linewidth=1.5, linestyle='--', label='Perfect forecast')
        ax.set_title('H1: Actual vs Predicted (non-zero demand)', fontsize=13)
        ax.set_xlabel('Actual Demand')
        ax.set_ylabel('Predicted Demand')
        r2 = all_results[1]['metrics']['r2']
        ax.text(0.05, 0.92, f'R\u00b2 = {r2:.3f}', transform=ax.transAxes, fontsize=12, color=GREEN)
        ax.legend()
        ax.grid(True)
        plt.tight_layout()
        plt.savefig(output_path / 'perf_03_actual_vs_predicted_h1.png', dpi=150, bbox_inches='tight')
        plt.close()

    # Plot 4: Residual distribution (H1)
    if 'y_test' in all_results[1]:
        errors = all_results[1]['y_test'] - all_results[1]['y_pred']
        nonzero_err = errors[all_results[1]['y_test'] > 0]
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.hist(np.clip(nonzero_err, -500, 500), bins=80, color=BLUE, alpha=0.8, edgecolor='#0f172a')
        ax.axvline(0, color=ORANGE, linewidth=2, linestyle='--', label='Zero error')
        ax.axvline(nonzero_err.mean(), color=GREEN, linewidth=1.5, linestyle='--',
                   label=f'Mean bias: {nonzero_err.mean():+.1f}')
        ax.set_title('H1: Forecast Error Distribution (Actual \u2212 Predicted)', fontsize=13)
        ax.set_xlabel('Error (units)')
        ax.set_ylabel('Frequency')
        ax.legend()
        ax.grid(axis='y')
        plt.tight_layout()
        plt.savefig(output_path / 'perf_04_residuals_h1.png', dpi=150, bbox_inches='tight')
        plt.close()

    # Plot 5: Bias and R² by horizon
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
    biases = [all_results[h]['metrics']['bias'] for h in horizons]
    r2s    = [all_results[h]['metrics']['r2']   for h in horizons]
    bias_colors = [GREEN if b >= 0 else ORANGE for b in biases]
    ax1.bar([f'H{h}' for h in horizons], biases, color=bias_colors, alpha=0.85, width=0.4)
    ax1.axhline(0, color='white', linewidth=1, linestyle='--')
    ax1.set_title('Forecast Bias by Horizon\n(+ = under-forecast)', fontsize=12)
    ax1.set_ylabel('Mean Error (units)')
    ax1.grid(axis='y')
    ax2.bar([f'H{h}' for h in horizons], r2s, color=PURPLE, alpha=0.85, width=0.4)
    ax2.set_title('R\u00b2 by Horizon\n(1.0 = perfect)', fontsize=12)
    ax2.set_ylabel('R\u00b2')
    ax2.set_ylim(0, 1)
    ax2.grid(axis='y')
    for i, v in enumerate(r2s):
        ax2.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)
    plt.tight_layout()
    plt.savefig(output_path / 'perf_05_bias_and_r2.png', dpi=150, bbox_inches='tight')
    plt.close()

    logger.info(f'\u2713 Performance plots saved to {output_path}')


def create_shap_report(model, X_test_sample, feature_cols, output_dir):
    """Generate SHAP summary plot for model explainability."""
    if not SHAP_AVAILABLE:
        return None
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)
    logger.info('Generating SHAP report...')
    try:
        explainer   = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_test_sample)
        if isinstance(shap_values, list):
            shap_values = shap_values[0]
        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_values, X_test_sample, show=False, max_display=15)
        plt.tight_layout()
        plt.savefig(output_path / 'shap_summary.png', dpi=100, bbox_inches='tight')
        plt.close()
        logger.info('\u2713 SHAP report complete')
    except Exception as e:
        logger.warning(f'SHAP failed: {e}')

print('Visualisation and SHAP functions defined.')


Visualisation and SHAP functions defined.


## 9. Multi-Horizon Model Training

Three separate LightGBM models are trained — one per forecast horizon.
Each model predicts demand `h` days ahead.

### Why separate models per horizon?
A single model predicting all horizons simultaneously (multi-output) tends to underfit
longer horizons. Separate models allow each to learn horizon-specific patterns:
- **H1**: Captures day-of-week effects, recent velocity, promo flags
- **H7**: Captures weekly seasonality, upcoming festival windows
- **H14**: Captures bi-weekly cycles, lead-time planning signals

### Target Construction
For horizon `h`, the target is `true_demand` shifted back by `h` days:
```python
target_h = df.groupby(['sku_id', 'location_id'])['true_demand'].shift(-h)
```
Rows where the shifted target is NaN (last `h` rows per group) are dropped.

### Train/Val/Test Split
Split is **temporal** (not random) to prevent future leakage:
- Train: first 70% of dates
- Validation: next 15% (used for early stopping)
- Test: final 15% (held-out evaluation)

All SKUs share the same date boundaries — this tests generalisation across the full
product catalogue on the same time period.

### LightGBM Hyperparameters
| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `objective` | `tweedie` | FMCG demand is overdispersed (many zeros, occasional spikes) |
| `tweedie_variance_power` | 1.5 | Balanced between Poisson (1.0) and Gamma (2.0) |
| `n_estimators` | 3000 | High ceiling; early stopping prevents overfitting |
| `learning_rate` | 0.03 | Low LR + many trees = better generalisation |
| `num_leaves` | 128 | Sufficient capacity for 50+ features |
| `min_child_samples` | 50 | Prevents overfitting on rare SKU-location combos |
| `early_stopping` | 100 rounds | Stops if val loss doesn't improve for 100 rounds |

### Memory Management
Each horizon's training data is deleted immediately after training (`del train_df, val_df, ...`).
Without this, three copies of the dataset would coexist in memory simultaneously.


In [10]:
def train_horizon_models(df, feature_cols, horizons=[1, 7, 14]):
    logger.info('\n' + '='*70)
    logger.info('TRAINING MULTI-HORIZON MODELS')
    logger.info('='*70)

    all_results = {}
    drift_baseline_captured = False

    df_sorted = df.sort_values(['sku_id', 'location_id', 'date']).reset_index(drop=True)

    for h in horizons:
        logger.info(f'\n{"="*70}')
        logger.info(f'TRAINING HORIZON h={h} MODEL')
        logger.info(f'{"="*70}')
        logger.info(f'Memory usage: {get_memory_usage():.1f} MB')

        # Build h-step-ahead target without copying the full dataframe
        target_col_h = f'target_h{h}'
        target_series = df_sorted.groupby(['sku_id', 'location_id'], observed=True)[TARGET_COL].shift(-h)
        valid_mask = target_series.notna()
        df_h = df_sorted[valid_mask].copy()
        df_h[target_col_h] = target_series[valid_mask].values
        del target_series, valid_mask
        gc.collect()

        # Temporal split — same date boundaries for all SKUs
        dates        = df_h['date'].sort_values().unique()
        train_cutoff = dates[int(len(dates) * 0.70)]
        val_cutoff   = dates[int(len(dates) * 0.85)]

        train_df = df_h[df_h['date'] <  train_cutoff].copy()
        val_df   = df_h[(df_h['date'] >= train_cutoff) & (df_h['date'] < val_cutoff)].copy()
        test_df  = df_h[df_h['date'] >= val_cutoff].copy()
        del df_h
        gc.collect()

        logger.info(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

        X_train = train_df[feature_cols]
        y_train = train_df[target_col_h].astype(np.float32)
        X_val   = val_df[feature_cols]
        y_val   = val_df[target_col_h].astype(np.float32)
        X_test  = test_df[feature_cols]
        y_test  = test_df[target_col_h].astype(np.float32)

        params = {
            'objective':               'tweedie',
            'tweedie_variance_power':  1.5,
            'metric':                  'tweedie',
            'n_estimators':            3000,
            'learning_rate':           0.03,
            'num_leaves':              128,
            'max_depth':               8,
            'min_child_samples':       50,
            'subsample':               0.75,
            'colsample_bytree':        0.75,
            'reg_lambda':              1.0,
            'reg_alpha':               0.5,
            'random_state':            42,
            'n_jobs':                  -1,
            'verbose':                 -1
        }

        logger.info(f'Training h={h} model...')
        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(100, verbose=False)]
        )

        y_test_pred = np.clip(model.predict(X_test), 0, None)
        metrics     = calculate_metrics(y_test.values, y_test_pred, label=f'H{h} Test Set')

        model_path = OUTPUT_ROOT / f'model_h{h}.txt'
        model.booster_.save_model(str(model_path))
        logger.info(f'\u2713 Model saved to {model_path}')

        all_results[h] = {
            'model':      model,
            'metrics':    metrics,
            'test_wmape': metrics['wmape'],
            'test_mae':   metrics['mae'],
            'y_pred':     y_test_pred,
            'y_test':     y_test.values.copy(),
            'test_df':    test_df[['date', 'sku_id', 'location_id']].copy()
        }

        # Capture drift baseline from h=1 training data (before it's deleted)
        if h == 1 and not drift_baseline_captured:
            try:
                detector    = DriftDetector(drift_threshold=0.15)
                train_sample = train_df.sample(min(1000, len(train_df)), random_state=42)
                detector.capture_baseline(
                    train_sample[feature_cols],
                    train_sample[target_col_h],
                    critical_features=feature_cols
                )
                detector.save_baseline(str(OUTPUT_ROOT / 'drift_baseline.json'))
                drift_baseline_captured = True
                logger.info('\u2713 Drift baseline captured')
            except Exception as e:
                logger.warning(f'Drift baseline failed: {e}')

        # Free memory immediately — critical for Kaggle's 16 GB limit
        del train_df, val_df, test_df, X_train, y_train, X_val, y_val, X_test, y_test
        gc.collect()

        # SHAP explainability — h=1 only (most actionable horizon)
        if h == 1 and ENABLE_SHAP:
            try:
                shap_output  = OUTPUT_ROOT / 'shap_output'
                X_test_sample = df_sorted[feature_cols].sample(min(SHAP_SAMPLE_TEST, len(df_sorted)), random_state=42)
                create_shap_report(model, X_test_sample, feature_cols, str(shap_output))
                del X_test_sample
                gc.collect()
            except Exception as e:
                logger.warning(f'SHAP failed: {e}')

    # Save feature importance (from h=1 model)
    fi = pd.DataFrame({
        'feature':    feature_cols,
        'importance': all_results[1]['model'].feature_importances_
    }).sort_values('importance', ascending=False)
    fi.to_csv(OUTPUT_ROOT / 'feature_importance.csv', index=False)

    # Save metrics summary
    metrics_rows = [{'horizon': h, **all_results[h]['metrics']} for h in horizons]
    pd.DataFrame(metrics_rows).to_csv(OUTPUT_ROOT / 'metrics_summary.csv', index=False)
    logger.info('\u2713 Metrics summary saved.')

    # Generate performance plots
    try:
        plot_model_performance(all_results, horizons, str(OUTPUT_ROOT / 'performance_plots'))
    except Exception as e:
        logger.warning(f'Performance plots failed: {e}')

    logger.info('\n' + '='*70)
    logger.info('MULTI-HORIZON TRAINING COMPLETE')
    logger.info('='*70)
    for h in horizons:
        logger.info(f'  h={h:2d} day: WMAPE = {all_results[h]["test_wmape"]:.2f}%')
    logger.info(f'Memory usage: {get_memory_usage():.1f} MB')
    logger.info('='*70)

    gc.collect()
    return all_results

print('Training function defined.')


Training function defined.


## 10. Main Pipeline

Orchestrates all steps end-to-end:

```
Load Data → Stockout Correction → Feature Engineering
    → Select Features → Train Models → Save Outputs
```

Memory checkpoints are logged at each step so you can identify where RAM spikes occur.

After feature engineering, the dataframe is trimmed to only the columns needed for training
(`feature_cols + TARGET_COL + id_cols`). This alone can save 30–50% memory.


In [11]:
def main():
    logger.info('='*70)
    logger.info('FMCG TRAINING PIPELINE (RICH FEATURES + OPTIMIZED)')
    logger.info('='*70)
    logger.info(f'Initial memory: {get_memory_usage():.1f} MB')

    # Step 1: Load
    data_path = DATA_ROOT / DAILY_TS_FILE
    logger.info(f'\nLoading data from {data_path}')
    df = pd.read_parquet(data_path)
    df['date'] = pd.to_datetime(df['date'])
    # Drop internal parquet metadata columns if present
    drop_prefixes = ['__fragment', '__batch', '__last', '__filename']
    df = df.drop(columns=[c for c in df.columns if any(c.startswith(p) for p in drop_prefixes)], errors='ignore')
    df = reduce_mem_usage(df, verbose=True)
    gc.collect()
    logger.info(f'Loaded {len(df):,} rows | Memory: {get_memory_usage():.1f} MB')

    # Step 2: Stockout Correction
    df = correct_stockouts(df)
    gc.collect()

    # Step 3: Feature Engineering
    df = engineer_features(df)
    df = reduce_mem_usage(df, verbose=True)
    gc.collect()
    logger.info(f'After feature engineering | Memory: {get_memory_usage():.1f} MB')

    # Step 4: Select Features
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    id_cols      = ['date', 'sku_id', 'location_id']
    exclude      = [TARGET_COL] + LEAK_COLS + id_cols
    feature_cols = [c for c in numeric_cols if c not in exclude and c in df.columns]
    logger.info(f'Using {len(feature_cols)} features')

    # Trim to only training columns to free memory
    keep = feature_cols + [TARGET_COL, 'date', 'sku_id', 'location_id']
    df   = df[[c for c in keep if c in df.columns]].copy()
    gc.collect()
    logger.info(f'Trimmed to training columns | Memory: {get_memory_usage():.1f} MB')

    # Step 5: Train
    horizon_results = train_horizon_models(df, feature_cols, horizons=HORIZONS)

    del df
    gc.collect()

    # Summary
    logger.info('\n' + '='*70)
    logger.info('PIPELINE COMPLETE')
    logger.info('='*70)
    for h in HORIZONS:
        logger.info(f'  h={h:2d} day: WMAPE = {horizon_results[h]["test_wmape"]:.2f}%')
    logger.info(f'Final memory: {get_memory_usage():.1f} MB')
    logger.info('='*70)

    return {
        'train_results':   horizon_results[1],
        'horizon_results': horizon_results,
        'feature_cols':    feature_cols
    }

print('Main pipeline defined.')


Main pipeline defined.


## 11. Inference Only (Pre-trained Model)

Use this cell when you want to generate forecasts without retraining.
Loads the saved `model_h1.txt` and `feature_importance.csv` from a previous run.

Useful for:
- Daily production inference on Kaggle
- Testing the pipeline without the 30–60 min training cost
- Debugging feature engineering changes


In [12]:
def forecast_only():
    """Load a pre-trained model and run inference — no retraining."""
    logger.info('='*70)
    logger.info('MULTI-STEP FORECASTING (USING SAVED MODEL)')
    logger.info('='*70)

    model_path = OUTPUT_ROOT / 'model_h1.txt'
    if not model_path.exists():
        logger.error(f'Model file not found: {model_path}')
        return None

    model = lgb.Booster(model_file=str(model_path))
    logger.info(f'Loaded model from {model_path}')

    fi_path = OUTPUT_ROOT / 'feature_importance.csv'
    if not fi_path.exists():
        logger.error(f'Feature importance file not found: {fi_path}')
        return None

    feature_cols = pd.read_csv(fi_path)['feature'].tolist()
    logger.info(f'Loaded {len(feature_cols)} features')

    df = pd.read_parquet(DATA_ROOT / DAILY_TS_FILE)
    df['date'] = pd.to_datetime(df['date'])
    df = correct_stockouts(df)
    df = engineer_features(df)
    df = reduce_mem_usage(df)

    logger.info('\u2713 Model loaded and data prepared. Ready for inference.')
    return model, feature_cols, df

print('forecast_only() defined.')


forecast_only() defined.


## 12. Run the Pipeline

Execute the full training pipeline. On Kaggle with GPU acceleration this takes ~30–45 minutes.
On CPU it may take 60–90 minutes depending on dataset size.

Outputs written to `OUTPUT_ROOT`:
```
model_h1.txt              # H1 LightGBM model
model_h7.txt              # H7 LightGBM model
model_h14.txt             # H14 LightGBM model
feature_importance.csv    # Feature importances from H1 model
metrics_summary.csv       # WMAPE/MAE/RMSE/R²/Bias per horizon
drift_baseline.json       # Training distribution baseline for drift monitoring
performance_plots/        # 5 diagnostic PNG charts
shap_output/              # SHAP summary plot (if SHAP available)
```


In [13]:
# Run the full training pipeline
# To run inference only on a pre-trained model, call forecast_only() instead

results = main()


2026-05-05 03:12:28,699 - INFO - ======================================================================
2026-05-05 03:12:28,699 - INFO - FMCG TRAINING PIPELINE (RICH FEATURES + OPTIMIZED)
2026-05-05 03:12:28,700 - INFO - ======================================================================
2026-05-05 03:12:28,701 - INFO - Initial memory: 617.4 MB
2026-05-05 03:12:28,702 - INFO - 
Loading data from /kaggle/input/datasets/sunchoehprince/fmcgparquet/daily_timeseries.parquet
2026-05-05 03:12:45,233 - INFO - Memory: 5740.47 MB → 3371.83 MB (41.3% reduction)
2026-05-05 03:12:45,370 - INFO - Loaded 14,610,000 rows | Memory: 10626.8 MB
2026-05-05 03:12:45,370 - INFO - Applying stockout correction...
2026-05-05 03:13:12,901 - INFO - Excluded 0 fully censored rows
2026-05-05 03:13:13,616 - INFO - Starting feature engineering...
2026-05-05 03:13:13,617 - INFO - SKU fields already present — skipping
2026-05-05 03:13:13,617 - INFO - Location fields already present — skipping
2026-05-05 03:13:17,74